In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [3]:
df = pd.read_csv('../../../week1/preprocessing/daily_sales.csv')

df = df[['Order Date', 'Total amount']]

df['Order Date'] = pd.to_datetime(df['Order Date'], format="mixed", dayfirst=True, errors="coerce")

df = (
    df.groupby('Order Date')['Total amount']
      .sum()
      .reset_index()
      .sort_values('Order Date')
      .reset_index(drop=True)
)

df.head()

,Order Date,Total amount
0,2011-01-04,16.448
1,2011-01-05,288.060
2,2011-01-06,19.536
3,2011-01-07,4407.100
4,2011-01-08,87.158


In [4]:
df['day'] = df['Order Date'].dt.day
df['month'] = df['Order Date'].dt.month
df['year'] = df['Order Date'].dt.year
df['dayofweek'] = df['Order Date'].dt.dayofweek
df['weekofyear'] = df['Order Date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['Order Date'].dt.quarter

df['lag1'] = df['Total amount'].shift(1)
df['lag7'] = df['Total amount'].shift(7)
df['lag30'] = df['Total amount'].shift(30)

df['rolling7'] = df['Total amount'].rolling(7).mean()
df['rolling30'] = df['Total amount'].rolling(30).mean()

df = df.dropna().reset_index(drop=True)

df.head()

,Order Date,Total amount,day,month,year,dayofweek,weekofyear,quarter,lag1,lag7,lag30,rolling7,rolling30
0,2011-02-13,129.568,13,2,2011,6,6,1,2043.400,211.646,16.448,418.550857,594.361633
1,2011-02-15,576.726,15,2,2011,1,7,1,129.568,97.112,288.060,487.067143,603.983833
2,2011-02-16,21.360,16,2,2011,2,7,1,576.726,134.384,19.536,470.920857,604.044633
3,2011-02-17,9.040,17,2,2011,3,7,1,21.360,330.512,4407.100,424.996286,457.442633
4,2011-02-18,54.208,18,2,2011,4,7,1,9.040,180.320,87.158,406.980286,456.344300


In [5]:
features = [
    'day', 'month', 'year', 'dayofweek',
    'weekofyear', 'quarter',
    'lag1', 'lag7', 'lag30',
    'rolling7', 'rolling30'
]

X = df[features]
y = df['Total amount']

split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

test_dates = df['Order Date'].iloc[split:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 966
Testing samples: 242


In [6]:
model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=1,
    gamma=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    random_state=42
)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Model trained successfully!")

Model trained successfully!


In [8]:
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

MAE: 1607.18200726674
RMSE: 2303.4952552568748
